# 02 · Fit Generators

Fit all three generators on the 100k LAION sample and save their parameters.

In [ ]:
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))
from utils import read_fbin
from perturbation import PerturbationGenerator
from gmm_generator import GMMGenerator

Path("../data/models").mkdir(parents=True, exist_ok=True)

## Load sample

In [ ]:
sample = read_fbin("../data/laion_sample_100k.fbin")
print(f"Sample shape: {sample.shape}")

## Fit DisturbGen-style perturbation generator

In [ ]:
gen_perturb = PerturbationGenerator(n_clusters=1000, seed=42)
gen_perturb.fit(sample)
gen_perturb.save("../data/models/perturbation.npz")
print("Perturbation generator saved.")

## Fit diagonal-covariance GMM

In [ ]:
gen_gmm = GMMGenerator(n_components=1000, seed=42)
gen_gmm.fit(sample)
gen_gmm.save("../data/models/gmm.pkl")
print(f"GMM converged: {gen_gmm.gmm_.converged_}")

## Inspect fitted centroids

Compare centroid norms across generators as a sanity check.

In [ ]:
import matplotlib.pyplot as plt

perturb_centroid_norms = np.linalg.norm(gen_perturb.centroids_, axis=1)
gmm_centroid_norms = np.linalg.norm(gen_gmm.gmm_.means_, axis=1)

plt.figure(figsize=(6, 3))
plt.hist(perturb_centroid_norms, bins=40, alpha=0.6, label="Perturbation")
plt.hist(gmm_centroid_norms, bins=40, alpha=0.6, label="GMM")
plt.xlabel("centroid L2 norm")
plt.ylabel("count")
plt.title("Centroid norm distributions")
plt.legend()
plt.tight_layout()
plt.savefig("../results/02_centroid_norms.png", dpi=120)
plt.show()